# Bitcoin Historical Data

In [2]:
import pandas as pd
import sys

sys.path.append('..')  # allows Python to look one directory up (the repo root)
from constants import RANDOM_SEED
from utils import set_seed

df = pd.read_csv("D:/CitrusBits/pythonic-rebirth/datasets/btcusd_1-min_data.csv")

print(df.shape)
print(df.head())
print(df.dtypes)

df['Timestamp'] = pd.to_datetime(df['Timestamp'], unit='s')
print(df['Timestamp'].min(), "to", df['Timestamp'].max())

(7637868, 6)
    Timestamp  Open  High   Low  Close  Volume
0  1325376060  4.58  4.58  4.58   4.58     0.0
1  1325376120  4.58  4.58  4.58   4.58     0.0
2  1325376180  4.58  4.58  4.58   4.58     0.0
3  1325376240  4.58  4.58  4.58   4.58     0.0
4  1325376300  4.58  4.58  4.58   4.58     0.0
Timestamp      int64
Open         float64
High         float64
Low          float64
Close        float64
Volume       float64
dtype: object
2012-01-01 00:01:00 to 2026-07-10 01:48:00


In [3]:
print("Rows with Volume == 0:", (df['Volume'] == 0).sum())
print("Percent of total:", (df['Volume'] == 0).mean() * 100)

Rows with Volume == 0: 1312362
Percent of total: 17.18230794247819


In [4]:
# resampling to a bigger more coarser interval
df = df.set_index('Timestamp')

daily = df.resample('D').agg({
    'Open': 'first',
    'High': 'max',
    'Low': 'min',
    'Close': 'last',
    'Volume': 'sum'
})

daily = daily.dropna()  # drop any days with no trading activity at all
print(daily.shape)
print(daily.head())

(5305, 5)
            Open  High   Low  Close      Volume
Timestamp                                      
2012-01-01  4.58  5.00  4.58   5.00   21.602000
2012-01-02  5.00  5.00  5.00   5.00   19.048000
2012-01-03  5.00  5.32  5.00   5.29   88.037281
2012-01-04  5.29  5.57  4.93   5.57  107.233260
2012-01-05  5.57  6.65  5.57   6.65   94.801829


In [5]:
lookback = 3  # use the last 3 days' data to predict the next day's Close

for lag in range(1, lookback + 1):
    daily[f'Close_lag{lag}'] = daily['Close'].shift(lag)
    daily[f'Volume_lag{lag}'] = daily['Volume'].shift(lag)
    daily[f'High_lag{lag}'] = daily['High'].shift(lag)
    daily[f'Low_lag{lag}'] = daily['Low'].shift(lag)

daily = daily.dropna()
print(daily.shape)
print(daily.head())

(5302, 17)
            Open  High   Low  Close      Volume  Close_lag1  Volume_lag1  \
Timestamp                                                                  
2012-01-04  5.29  5.57  4.93   5.57  107.233260        5.29    88.037281   
2012-01-05  5.57  6.65  5.57   6.65   94.801829        5.57   107.233260   
2012-01-06  6.65  6.90  6.00   6.00   33.882747        6.65    94.801829   
2012-01-07  6.00  6.80  6.00   6.80    0.295858        6.00    33.882747   
2012-01-08  6.80  7.00  6.80   7.00    5.000000        6.80     0.295858   

            High_lag1  Low_lag1  Close_lag2  Volume_lag2  High_lag2  Low_lag2  \
Timestamp                                                                       
2012-01-04       5.32      5.00        5.00    19.048000       5.00      5.00   
2012-01-05       5.57      4.93        5.29    88.037281       5.32      5.00   
2012-01-06       6.65      5.57        5.57   107.233260       5.57      4.93   
2012-01-07       6.90      6.00        6.65    94.8

In [7]:
# separting features x and target y
feature_cols = [col for col in daily.columns if 'lag' in col]
X = daily[feature_cols]
y = daily['Close']

print(X.columns.tolist())
print(X.shape, y.shape)

['Close_lag1', 'Volume_lag1', 'High_lag1', 'Low_lag1', 'Close_lag2', 'Volume_lag2', 'High_lag2', 'Low_lag2', 'Close_lag3', 'Volume_lag3', 'High_lag3', 'Low_lag3']
(5302, 12) (5302,)


In [8]:
n = len(daily)
train_end = int(n * 0.70)
val_end = int(n * 0.80)

X_train = X.iloc[:train_end]
y_train = y.iloc[:train_end]

X_val = X.iloc[train_end:val_end]
y_val = y.iloc[train_end:val_end]

X_test = X.iloc[val_end:]
y_test = y.iloc[val_end:]

print("Train:", X_train.shape, X_train.index.min(), "to", X_train.index.max())
print("Val:", X_val.shape, X_val.index.min(), "to", X_val.index.max())
print("Test:", X_test.shape, X_test.index.min(), "to", X_test.index.max())

Train: (3711, 12) 2012-01-04 00:00:00 to 2022-03-02 00:00:00
Val: (530, 12) 2022-03-03 00:00:00 to 2023-08-14 00:00:00
Test: (1061, 12) 2023-08-15 00:00:00 to 2026-07-10 00:00:00


In [9]:
# scaling features
# fit on train only

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_val_scaled = pd.DataFrame(scaler.transform(X_val), columns=X_val.columns, index=X_val.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

print(X_train_scaled.describe())

         Close_lag1   Volume_lag1     High_lag1     Low_lag1    Close_lag2  \
count  3.711000e+03  3.711000e+03  3.711000e+03  3711.000000  3.711000e+03   
mean  -3.063510e-17 -6.127019e-17 -3.063510e-17     0.000000  3.063510e-17   
std    1.000135e+00  1.000135e+00  1.000135e+00     1.000135  1.000135e+00   
min   -5.731971e-01 -9.541745e-01 -5.732798e-01    -0.572859 -5.728206e-01   
25%   -5.546608e-01 -6.030178e-01 -5.549805e-01    -0.553936 -5.542701e-01   
50%   -5.048753e-01 -2.715824e-01 -5.053008e-01    -0.505451 -5.047283e-01   
75%    3.137231e-02  2.670867e-01  3.226124e-02     0.035962  3.218577e-02   
max    4.020900e+00  1.300229e+01  3.990907e+00     4.094623  4.024791e+00   

        Volume_lag2     High_lag2     Low_lag2    Close_lag3   Volume_lag3  \
count  3.711000e+03  3.711000e+03  3711.000000  3.711000e+03  3.711000e+03   
mean  -6.509958e-17 -3.063510e-17     0.000000  6.127019e-17 -4.595264e-17   
std    1.000135e+00  1.000135e+00     1.000135  1.000135e+00  1

In [10]:
# decision tree via sklearn

import joblib
import os
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score

model_dir = "D:/CitrusBits/pythonic-rebirth/models"
os.makedirs(model_dir, exist_ok=True)
dt_path = os.path.join(model_dir, "btc_decision_tree.pkl")

if os.path.exists(dt_path):
    print("Loading existing Decision Tree model...")
    dt_model = joblib.load(dt_path)
else:
    print("Training new Decision Tree model...")
    dt_model = DecisionTreeRegressor(random_state=RANDOM_SEED, max_depth=8)
    dt_model.fit(X_train_scaled, y_train)
    joblib.dump(dt_model, dt_path)
    print("Model saved.")

dt_val_preds = dt_model.predict(X_val_scaled)
dt_val_mse = mean_squared_error(y_val, dt_val_preds)
dt_val_r2 = r2_score(y_val, dt_val_preds)

print(f"Decision Tree (Validation) → MSE: {dt_val_mse:.4f}, R²: {dt_val_r2:.4f}")

Training new Decision Tree model...
Model saved.
Decision Tree (Validation) → MSE: 3407358.2976, R²: 0.9360


In [11]:
importances = pd.Series(dt_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(importances)

Close_lag1     0.994143
Low_lag1       0.003958
High_lag1      0.001036
Volume_lag1    0.000207
Volume_lag3    0.000141
High_lag2      0.000121
Volume_lag2    0.000114
Close_lag2     0.000086
Low_lag3       0.000075
Close_lag3     0.000051
Low_lag2       0.000049
High_lag3      0.000020
dtype: float64


In [12]:
# doing pytorch based linear regression

import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1, 1)).flatten()
y_val_scaled = y_scaler.transform(y_val.values.reshape(-1, 1)).flatten()

X_train_tensor = torch.tensor(X_train_scaled.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32).view(-1, 1)
X_val_tensor = torch.tensor(X_val_scaled.values, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val_scaled, dtype=torch.float32).view(-1, 1)


class LinearRegressionModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        return self.linear(x)


torch_model_path = os.path.join(model_dir, "btc_linear_regression_torch.pth")

if os.path.exists(torch_model_path):
    os.remove(torch_model_path)  # fresh start given past stale-model issues

torch.manual_seed(RANDOM_SEED)
lr_model_torch = LinearRegressionModel(X_train_scaled.shape[1])

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(lr_model_torch.parameters(), lr=0.01)

epochs = 1000
train_losses = []
val_losses = []

for epoch in range(epochs):
    lr_model_torch.train()
    optimizer.zero_grad()
    predictions = lr_model_torch(X_train_tensor)
    loss = criterion(predictions, y_train_tensor)
    loss.backward()
    optimizer.step()
    train_losses.append(loss.item())

    lr_model_torch.eval()
    with torch.no_grad():
        val_preds = lr_model_torch(X_val_tensor)
        val_loss = criterion(val_preds, y_val_tensor)
        val_losses.append(val_loss.item())

    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch + 1}/{epochs} | Train Loss: {loss.item():.4f} | Val Loss: {val_loss.item():.4f}")

torch.save(lr_model_torch.state_dict(), torch_model_path)

Epoch 100/1000 | Train Loss: 0.0031 | Val Loss: 0.0048
Epoch 200/1000 | Train Loss: 0.0027 | Val Loss: 0.0040
Epoch 300/1000 | Train Loss: 0.0024 | Val Loss: 0.0035
Epoch 400/1000 | Train Loss: 0.0023 | Val Loss: 0.0032
Epoch 500/1000 | Train Loss: 0.0023 | Val Loss: 0.0031
Epoch 600/1000 | Train Loss: 0.0022 | Val Loss: 0.0031
Epoch 700/1000 | Train Loss: 0.0022 | Val Loss: 0.0030
Epoch 800/1000 | Train Loss: 0.0022 | Val Loss: 0.0030
Epoch 900/1000 | Train Loss: 0.0022 | Val Loss: 0.0030
Epoch 1000/1000 | Train Loss: 0.0022 | Val Loss: 0.0029


In [13]:
lr_model_torch.eval()
with torch.no_grad():
    val_preds_scaled_output = lr_model_torch(X_val_tensor).numpy().flatten()

val_preds_actual = y_scaler.inverse_transform(val_preds_scaled_output.reshape(-1, 1)).flatten()

lr_val_mse_torch = mean_squared_error(y_val, val_preds_actual)
lr_val_r2_torch = r2_score(y_val, val_preds_actual)

print(f"PyTorch Linear Regression (Validation) → MSE: {lr_val_mse_torch:.4f}, R²: {lr_val_r2_torch:.4f}")

PyTorch Linear Regression (Validation) → MSE: 636921.1833, R²: 0.9880


In [14]:
print(f"{'Model':<30}{'Val MSE':<15}{'Val R²':<10}")
print(f"{'PyTorch Linear Regression':<30}{636921.1833:<15.4f}{0.9880:<10.4f}")
print(f"{'Decision Tree':<30}{3407358.2976:<15.4f}{0.9360:<10.4f}")

Model                         Val MSE        Val R²    
PyTorch Linear Regression     636921.1833    0.9880    
Decision Tree                 3407358.2976   0.9360    
